# Fase 7 — Comunicación de resultados (Flujo E)

| Campo | Valor |
|---|---|
| **Rol líder** | Todos los roles |
| **Entradas** | Resultados de Fases 1-6 |
| **Salidas** | Tablas y figuras finales para el artículo IEEE, video y slides |
| **Doc de fase** | [`docs/analisis_proyecto/fases/fase7_comunicacion.md`](../../docs/analisis_proyecto/fases/fase7_comunicacion.md) |

Este notebook **no** redacta el artículo: consolida los resultados numéricos en formato listo para LaTeX/Markdown, genera las figuras finales y compila un caso de estudio. Las tablas se exportan a CSV en `outputs/reports/` para que el equipo de redacción las cite directamente.


## 0. Configuración

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'analisis_proyecto').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPORTS_DIR = ROOT / 'outputs' / 'reports'
FIG_DIR = ROOT / 'outputs' / 'reports' / 'figures'
for d in (REPORTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
print(f'ROOT: {ROOT}')
print(f'REPORTS_DIR: {REPORTS_DIR.relative_to(ROOT)}')

## 1. Tabla 1 — Comparativa Tox21 (GIN vs baselines)

Se construye desde `outputs/results/gin_results.csv` y `outputs/results/baseline_results.csv`.

In [ ]:
results_dir = ROOT / 'outputs' / 'results'
tabla1_rows = []

baseline_csv = results_dir / 'baseline_results.csv'
gin_csv = results_dir / 'gin_results.csv'

if baseline_csv.exists():
    bdf = pd.read_csv(baseline_csv)
    if {'model', 'auc'}.issubset(bdf.columns):
        for model_name, group in bdf.groupby('model'):
            tabla1_rows.append({
                'modelo': model_name,
                'auc_promedio': group['auc'].mean(),
                'mejor_tarea': group.sort_values('auc', ascending=False)['task'].iloc[0]
                                if 'task' in group.columns else '—',
                'fuente': baseline_csv.name,
            })

if gin_csv.exists():
    gdf = pd.read_csv(gin_csv)
    if 'auc' in gdf.columns:
        tabla1_rows.append({
            'modelo': 'GNN-GIN',
            'auc_promedio': gdf['auc'].mean(),
            'mejor_tarea': gdf.sort_values('auc', ascending=False)['task'].iloc[0]
                            if 'task' in gdf.columns else '—',
            'fuente': gin_csv.name,
        })

tabla1 = pd.DataFrame(tabla1_rows)
if not tabla1.empty:
    tabla1['auc_promedio'] = tabla1['auc_promedio'].round(4)
    tabla1.to_csv(REPORTS_DIR / 'tabla1_modelos_tox21.csv', index=False)
    display(tabla1.sort_values('auc_promedio', ascending=False))
else:
    print('Sin resultados Tox21 — corre Fases 2 (baselines) y 3 (GIN) del proyecto JIC.')

## 2. Tabla 2 — Impacto del split (ChEMBL)

Filas vs compuesto, lectura directa de `metrics_summary.csv`.

In [ ]:
metrics_path = ROOT / 'outputs' / 'chembl' / 'results' / 'metrics_summary.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    cls_metrics = metrics[metrics['tarea'] == 'clasificacion']
    pivot = (cls_metrics
             .pivot_table(index='modelo', columns='split', values='accuracy_test', aggfunc='mean')
             .round(4))
    pivot['delta_filas_minus_compuesto'] = (pivot.get('filas') - pivot.get('compuesto')).round(4)
    pivot.to_csv(REPORTS_DIR / 'tabla2_split_impact.csv')
    display(pivot)
    print('\nNota: la diferencia positiva grande evidencia data leakage por filas.')
else:
    print('Sin metrics_summary.csv — corre `fase4_modelado.ipynb`.')

## 3. Tabla 3 — Exposición por provincia

In [ ]:
inec_csv = ROOT / 'data' / 'raw' / 'inec_sociodemografico.csv'
if inec_csv.exists():
    inec = pd.read_csv(inec_csv)
    cols = [c for c in ['poblacion', 'superficie_agricola_ha', 'indice_pobreza',
                        'area_km2', 'exposure_risk'] if c in inec.columns]
    if cols and 'provincia' in inec.columns:
        tabla3 = (inec.groupby('provincia')[cols]
                      .mean(numeric_only=True)
                      .round(3)
                      .sort_values(cols[0], ascending=False))
        tabla3.to_csv(REPORTS_DIR / 'tabla3_exposicion_provincia.csv')
        display(tabla3)
    else:
        print('CSV INEC sin columnas esperadas — verifica Fase 6.')
else:
    print('Sin CSV INEC — corre `fase6_geodatos.ipynb`.')

## 4. Figura combinada para el artículo (mosaico)

Resume gráficamente: AUC por modelo, importancia de features y mapa de exposición.

In [ ]:
fig, axes = plt.subplot_mosaic(
    [['auc', 'imp'], ['exposure', 'exposure']],
    figsize=(13, 9),
)

# Panel A — AUC por modelo
if not tabla1.empty:
    tabla1_sorted = tabla1.sort_values('auc_promedio', ascending=True)
    axes['auc'].barh(tabla1_sorted['modelo'], tabla1_sorted['auc_promedio'], color='steelblue')
    axes['auc'].set_title('Panel A — AUC promedio (Tox21)')
    axes['auc'].set_xlim(0.5, 1.0)
else:
    axes['auc'].text(0.5, 0.5, 'Sin datos Tox21', ha='center', va='center')
    axes['auc'].axis('off')

# Panel B — Importancia de features (ChEMBL)
imp_csv = ROOT / 'outputs' / 'chembl' / 'figures' / 'feature_importance_rf.png'
models_pkl = ROOT / 'outputs' / 'chembl' / 'models' / 'rf_classifier.pkl'
if models_pkl.exists():
    import joblib
    rf = joblib.load(models_pkl)
    feature_cols = json.loads((ROOT / 'outputs' / 'chembl' / 'models' / 'feature_cols.json').read_text(encoding='utf-8'))
    imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
    imp.plot(kind='barh', ax=axes['imp'], color='teal')
    axes['imp'].set_title('Panel B — Importancia features (RF)')
else:
    axes['imp'].text(0.5, 0.5, 'rf_classifier.pkl ausente', ha='center', va='center')
    axes['imp'].axis('off')

# Panel C — Exposición por provincia
if 'tabla3' in dir() and not tabla3.empty:
    plot_col = tabla3.columns[0]
    tabla3_sorted = tabla3[plot_col].sort_values(ascending=True)
    axes['exposure'].barh(tabla3_sorted.index, tabla3_sorted.values, color='salmon')
    axes['exposure'].set_title(f'Panel C — {plot_col} por provincia')
else:
    axes['exposure'].text(0.5, 0.5, 'Sin geodatos', ha='center', va='center')
    axes['exposure'].axis('off')

plt.tight_layout()
out = FIG_DIR / 'fig_mosaico_articulo.png'
fig.savefig(out, dpi=140, bbox_inches='tight')
print(f'guardado: {out.relative_to(ROOT)}')
plt.show()

## 5. Caso de estudio — Clorpirifos

Recopila evidencia transversal: registros ChEMBL, predicción Tox21 (si existe) y disponibilidad de explicación XAI.

In [ ]:
case_compound = 'Chlorpyrifos'
clean_csv = ROOT / 'data' / 'processed' / 'chembl_clean.csv'
predictions_csv = ROOT / 'outputs' / 'results' / 'panama_predictions.csv'
xai_dir = ROOT / 'outputs' / 'xai' / 'figures'

case_lines = [f'### Caso de estudio: {case_compound}']

if clean_csv.exists():
    df = pd.read_csv(clean_csv)
    sub = df[df['compound_name'].str.contains(case_compound, case=False, na=False)]
    case_lines.append(f'- Registros ChEMBL: **{len(sub)}**')
    if not sub.empty:
        case_lines.append(f'- pChEMBL medio: **{sub["pchembl_value"].mean():.2f}**')
        case_lines.append(f'- % Active: **{(sub["activity_class"] == "Active").mean() * 100:.1f}%**')
        case_lines.append(f'- standard_type principales: {dict(sub["standard_type"].value_counts().head(3))}')

if predictions_csv.exists():
    pred = pd.read_csv(predictions_csv)
    if 'compound_name' in pred.columns:
        sub_pred = pred[pred['compound_name'].str.contains(case_compound, case=False, na=False)]
        if not sub_pred.empty:
            case_lines.append(f'- Filas en predicciones GIN: **{len(sub_pred)}**')

xai_files = list(xai_dir.glob(f'*{case_compound.lower()}*')) if xai_dir.exists() else []
xai_files += list(xai_dir.glob(f'*{case_compound}*')) if xai_dir.exists() else []
case_lines.append(f'- SVG XAI disponibles: **{len(set(xai_files))}**')
for line in case_lines:
    print(line)
(REPORTS_DIR / 'caso_clorpirifos.md').write_text('\n'.join(case_lines), encoding='utf-8')

## 6. Resumen de hallazgos para Sección VI del artículo

In [ ]:
hallazgos = pd.DataFrame({
    'tema': [
        'Modelado clásico (ChEMBL)',
        'Split por filas vs por compuesto',
        'Modelo GIN sobre Tox21',
        'Interpretabilidad XAI',
        'Riesgo de exposición (Panamá)',
    ],
    'hallazgo': [
        'Descriptores moleculares globales (MW, LogP, PSA) tienen poder predictivo limitado: R² puede ser negativo en split honesto.',
        'La accuracy se infla ~40 puntos porcentuales con split por filas — necesario reportar ambos protocolos.',
        'GIN supera RF y MLP cuando hay señal específica de la diana; AUC promedio depende de la cobertura de tareas Tox21.',
        'GNNExplainer y Grad-CAM destacan grupos funcionales coherentes con literatura (P=S en organofosforados, triazol en azoles).',
        'Las provincias con mayor fracción agrícola (Herrera, Chiriquí) concentran el mayor índice compuesto de exposición.',
    ],
    'fuente': [
        'metrics_summary.csv (tabla 2)',
        'metrics_summary.csv',
        'gin_results.csv + baseline_results.csv (tabla 1)',
        'outputs/xai/figures + chemical_coherence.py',
        'inec_sociodemografico.csv (tabla 3)',
    ],
})
hallazgos.to_csv(REPORTS_DIR / 'hallazgos_resumen.csv', index=False)
display(hallazgos)

## 7. Checklist de entregables Fase 7

In [ ]:
expected_files = [
    REPORTS_DIR / 'tabla1_modelos_tox21.csv',
    REPORTS_DIR / 'tabla2_split_impact.csv',
    REPORTS_DIR / 'tabla3_exposicion_provincia.csv',
    REPORTS_DIR / 'caso_clorpirifos.md',
    REPORTS_DIR / 'hallazgos_resumen.csv',
    FIG_DIR / 'fig_mosaico_articulo.png',
]
checks = [
    (str(p.relative_to(ROOT)), p.exists(), f'{p.stat().st_size if p.exists() else 0} bytes')
    for p in expected_files
]
df_check = pd.DataFrame(checks, columns=['Entregable', 'Pasa', 'Detalle'])
df_check['Pasa'] = df_check['Pasa'].map({True: 'OK', False: 'FALTA'})
df_check

## 8. Próximos pasos

1. Importar `outputs/reports/tabla*.csv` en LaTeX (`\input{tabla1_modelos_tox21.csv}` con `csvsimple`).
2. Embeber `outputs/reports/figures/fig_mosaico_articulo.png` en la Sección IV del artículo IEEE.
3. Citar las **limitaciones honestas** documentadas en `docs/analisis_proyecto/fases/fase7_comunicacion.md` (sección 7).
4. Grabar el video siguiendo la estructura del doc de fase (5-10 min, demo del dashboard).
5. Producir slides reutilizando las figuras generadas aquí.

---
*Fase anterior:* [`fase6_geodatos.ipynb`](fase6_geodatos.ipynb)  
*Vuelta al inicio:* [`fase1_adquisicion.ipynb`](fase1_adquisicion.ipynb)